# DataHub Error Case 06 — MySQL Connection Error

**Monitor:** DataHub MySQL Connection Error  
**Component:** datahub-gms  

## ⚠️ WARNING
This test restarts MySQL → DataHub unavailable ~60-90 seconds

In [6]:
import sys
sys.path.insert(0, '.')
import importlib
import datahub_test_utils
importlib.reload(datahub_test_utils)
from datahub_test_utils import *

kubectl('get pods -n datahub-hynix | grep mysql')
r = health_check()
print(f'GMS health before test: {r.status_code}')

✅ DataHub test utils loaded
   GMS: http://192.168.139.2:8080
   OpenSearch: https://192.168.194.1:9200
prerequisites-mysql-0                                         1/1     Running     0               18h

GMS health before test: 200


In [7]:
# Cell 2: Restart MySQL
print('Restarting MySQL...')
kubectl('rollout restart statefulset/prerequisites-mysql -n datahub-hynix')
time.sleep(10)

for i in range(3):
    try:
        r = requests.get(f'{DATAHUB_URL}/health', timeout=5)
        print(f'Attempt {i+1}: {r.status_code}')
    except Exception as e:
        print(f'Attempt {i+1}: {e}')
    time.sleep(5)

Restarting MySQL...
statefulset.apps/prerequisites-mysql restarted

Attempt 1: 200
Attempt 2: 200
Attempt 3: 200


In [8]:
# Cell 3: Find MySQL error logs
wait_for_logs(15)
client = os_client()
result = client.search(
    index=LOG_INDEX,
    body={
        'query': {
            'bool': {
                'must': [
                    {'range': {'@timestamp': {'gte': 'now-15m'}}},
                    {'bool': {
                        'should': [
                            {'term': {'kubernetes.container_name.keyword': 'datahub-gms'}},
                            {'term': {'kubernetes.container_name.keyword': 'datahub-mae-consumer'}},
                        ],
                        'minimum_should_match': 1,
                    }},
                ],
                'should': [
                    {'match': {'message': 'Connection refused'}},
                    {'match': {'message': 'Cannot acquire'}},
                    {'match': {'message': 'MySQL'}},
                    {'match': {'message': 'ebean'}},
                ],
                'minimum_should_match': 1,
            }
        },
        'sort': [{'@timestamp': {'order': 'desc'}}],
        'size': 10,
    }
)
print_logs(result['hits']['hits'])

⏳ Waiting 15s for logs to reach OpenSearch...
✅ Found 2 log entries:

  [2026-06-11T01:36:14] ERROR | datahub-gms | io.ebean.datasource
         Connection [name[gmsEbeanDatabaseConfig3] startTime[1781141774814] busySeconds[0] createdBy[null] stmt[null]] not found 

  [2026-06-11T01:36:14] WARN  | datahub-gms | io.ebean.datasource
         DataSource closing busy connection? name[gmsEbeanDatabaseConfig3] startTime[1781141774814] busySeconds[0] stackTrace[] s



In [9]:
# Cell 4: Wait for recovery
print('Waiting for MySQL recovery...')
for i in range(9):
    time.sleep(10)
    r = health_check()
    print(f'GMS health: {r.status_code}')
    if r.status_code == 200:
        print('Recovery complete!')
        break

print('\n=== Summary ===')
print('Error:   MySQL connection refused')
print('Monitor: MySQL Connection Error fires on count > 0 in 1 min')
print('Fix:     kubectl rollout restart statefulset/prerequisites-mysql')

Waiting for MySQL recovery...
GMS health: 200
Recovery complete!

=== Summary ===
Error:   MySQL connection refused
Monitor: MySQL Connection Error fires on count > 0 in 1 min
Fix:     kubectl rollout restart statefulset/prerequisites-mysql
